<a href="https://colab.research.google.com/github/GangadharPrathap/MASAI-Assignments/blob/main/used_car_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Used Car Dataset Analysis
## Exploratory Data Analysis, Cleaning, and Baseline Model Evaluation

This notebook systematically explores, cleans, and evaluates a baseline model on a used car dataset.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error

# Set display options for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("Libraries imported successfully!")

---
## Task 1: Explore and Identify Issues

Load the dataset and identify at least three data quality problems.

In [ ]:
# Load the dataset
# Update the filename if your dataset has a different name
df = pd.read_csv('used_cars.csv')

print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")

In [ ]:
# Display first few rows
print("First 5 rows of the dataset:")
df.head()

In [ ]:
# Get dataset shape
print(f"Dataset Shape: {df.shape}")
print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")

In [ ]:
# Display dataset info
print("Dataset Information:")
df.info()

In [ ]:
# Display statistical summary
print("Statistical Summary:")
df.describe()

In [ ]:
# Check for missing values
print("Missing Values Count:")
missing_counts = df.isnull().sum()
missing_percentages = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing_counts,
    'Percentage': missing_percentages
})
print(missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False))

In [ ]:
# Check for duplicate rows
print(f"\nNumber of duplicate rows: {df.duplicated().sum()}")

In [ ]:
# Examine specific columns for data quality issues
print("\nUnique values in key columns:")
for col in df.columns:
    if df[col].dtype == 'object':
        print(f"\n{col}: {df[col].nunique()} unique values")
        print(f"Sample values: {df[col].unique()[:10]}")

---
## Task 2: Clean the Data

Systematically fix the identified issues.

In [ ]:
# Create a copy for cleaning
df_clean = df.copy()
print(f"Starting cleaning process with {len(df_clean)} rows")

### Step 1: Drop rows with null target values

In [ ]:
# Drop rows where selling_price is null
initial_rows = len(df_clean)
df_clean = df_clean.dropna(subset=['selling_price'])
rows_dropped = initial_rows - len(df_clean)
print(f"Dropped {rows_dropped} rows with null selling_price")
print(f"Remaining rows: {len(df_clean)}")

### Step 2: Clean and standardize the brand column

In [ ]:
# Strip whitespace and convert to lowercase
if 'brand' in df_clean.columns:
    print("Before cleaning:")
    print(df_clean['brand'].unique()[:20])

    df_clean['brand'] = df_clean['brand'].str.strip().str.lower()

    print("\nAfter cleaning:")
    print(df_clean['brand'].unique()[:20])
    print(f"\nUnique brands after cleaning: {df_clean['brand'].nunique()}")
else:
    print("Warning: 'brand' column not found in dataset")

### Step 3: Extract numeric values from mileage column

In [ ]:
# Extract numeric values from mileage column
if 'mileage' in df_clean.columns:
    print("Before extraction:")
    print(f"Data type: {df_clean['mileage'].dtype}")
    print(f"Sample values: {df_clean['mileage'].head(10).tolist()}")

    # Convert to string first, then extract numbers
    df_clean['mileage'] = df_clean['mileage'].astype(str)
    # Remove all non-numeric characters except decimal point
    df_clean['mileage'] = df_clean['mileage'].str.replace(r'[^0-9.]', '', regex=True)
    # Convert to numeric, invalid values become NaN
    df_clean['mileage'] = pd.to_numeric(df_clean['mileage'], errors='coerce')

    print("\nAfter extraction:")
    print(f"Data type: {df_clean['mileage'].dtype}")
    print(f"Sample values: {df_clean['mileage'].head(10).tolist()}")
    print(f"Missing values created: {df_clean['mileage'].isnull().sum()}")
else:
    print("Warning: 'mileage' column not found in dataset")

### Step 4: Remove duplicate rows

In [ ]:
# Remove duplicate rows
initial_rows = len(df_clean)
df_clean = df_clean.drop_duplicates()
duplicates_removed = initial_rows - len(df_clean)
print(f"Removed {duplicates_removed} duplicate rows")
print(f"Remaining rows: {len(df_clean)}")

### Step 5: Impute missing values in input features

In [ ]:
# Check remaining missing values
print("Missing values before imputation:")
print(df_clean.isnull().sum())

In [ ]:
# Impute missing values
# For numeric columns: use median
# For categorical columns: use mode

for col in df_clean.columns:
    if col == 'selling_price':  # Skip target variable
        continue

    if df_clean[col].isnull().sum() > 0:
        if df_clean[col].dtype in ['float64', 'int64']:
            # Numeric: impute with median
            median_value = df_clean[col].median()
            df_clean[col].fillna(median_value, inplace=True)
            print(f"Imputed {col} with median: {median_value}")
        else:
            # Categorical: impute with mode
            mode_value = df_clean[col].mode()[0] if not df_clean[col].mode().empty else 'unknown'
            df_clean[col].fillna(mode_value, inplace=True)
            print(f"Imputed {col} with mode: {mode_value}")

In [ ]:
# Verify no missing values remain (except possibly in target)
print("\nMissing values after imputation:")
print(df_clean.isnull().sum())

### Step 6: Remove impossible values (optional but recommended)

In [ ]:
# Remove rows with impossible values
initial_rows = len(df_clean)

# Remove negative or zero selling prices
df_clean = df_clean[df_clean['selling_price'] > 0]

# Remove negative mileage if column exists
if 'mileage' in df_clean.columns:
    df_clean = df_clean[df_clean['mileage'] >= 0]

# Remove unrealistic years if year column exists
if 'year' in df_clean.columns:
    current_year = 2024
    df_clean = df_clean[(df_clean['year'] >= 1900) & (df_clean['year'] <= current_year)]

rows_removed = initial_rows - len(df_clean)
print(f"Removed {rows_removed} rows with impossible values")
print(f"Final cleaned dataset: {len(df_clean)} rows")

In [ ]:
# Final check on cleaned data
print("\n=== CLEANED DATASET SUMMARY ===")
print(f"Shape: {df_clean.shape}")
print(f"\nMissing values:")
print(df_clean.isnull().sum())
print(f"\nData types:")
print(df_clean.dtypes)

---
## Task 3: Compute Baseline MAE

Build a baseline model that predicts the mean `selling_price` for every record and calculate MAE.

In [ ]:
# Calculate the mean selling price (baseline prediction)
mean_price = df_clean['selling_price'].mean()
print(f"Mean Selling Price (Baseline Prediction): ${mean_price:,.2f}")

In [ ]:
# Create baseline predictions (mean for all records)
baseline_predictions = np.full(len(df_clean), mean_price)
print(f"Number of predictions: {len(baseline_predictions)}")
print(f"All predictions are: ${baseline_predictions[0]:,.2f}")

In [ ]:
# Get actual selling prices
actual_prices = df_clean['selling_price'].values
print(f"Actual price range: ${actual_prices.min():,.2f} to ${actual_prices.max():,.2f}")

In [ ]:
# Calculate Mean Absolute Error (MAE)
baseline_mae = mean_absolute_error(actual_prices, baseline_predictions)

print("\n" + "="*60)
print("BASELINE MODEL EVALUATION")
print("="*60)
print(f"Model: Mean prediction for all records")
print(f"Prediction value: ${mean_price:,.2f}")
print(f"\nMean Absolute Error (MAE): ${baseline_mae:,.2f}")
print("="*60)

In [ ]:
# Additional metrics for context
print("\nAdditional Context:")
print(f"Mean Selling Price: ${mean_price:,.2f}")
print(f"Median Selling Price: ${df_clean['selling_price'].median():,.2f}")
print(f"Std Dev: ${df_clean['selling_price'].std():,.2f}")
print(f"MAE as % of mean: {(baseline_mae / mean_price) * 100:.2f}%")

In [ ]:
# Visualize the baseline performance
plt.figure(figsize=(12, 5))

# Plot 1: Distribution of actual prices vs baseline
plt.subplot(1, 2, 1)
plt.hist(actual_prices, bins=50, alpha=0.7, edgecolor='black', label='Actual Prices')
plt.axvline(mean_price, color='red', linestyle='--', linewidth=2, label=f'Baseline (Mean): ${mean_price:,.0f}')
plt.xlabel('Selling Price ($)')
plt.ylabel('Frequency')
plt.title('Distribution of Actual Prices vs Baseline Prediction')
plt.legend()
plt.grid(alpha=0.3)

# Plot 2: Residuals (errors)
plt.subplot(1, 2, 2)
residuals = actual_prices - baseline_predictions
plt.hist(residuals, bins=50, alpha=0.7, edgecolor='black', color='coral')
plt.axvline(0, color='red', linestyle='--', linewidth=2, label='Zero Error')
plt.xlabel('Prediction Error ($)')
plt.ylabel('Frequency')
plt.title(f'Baseline Model Residuals (MAE: ${baseline_mae:,.0f})')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()